In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import cross_validate, KFold

from catboost import CatBoostRegressor, cv, Pool

SEED = 1660

## Загружаем данные

In [ ]:
df_train = pd.read_parquet("03_datasets/final_train.parquet")
df_test = pd.read_parquet("03_datasets/final_test.parquet")

In [ ]:
X_train = df_train.drop(columns=["GameId", "Elo"])
Y_train = df_train["Elo"]

X_test = df_test.drop(columns=["GameId", "Elo"])
Y_test = df_test["Elo"]

In [ ]:
X_train.shape, X_test.shape

## Вспомогательные функции

In [ ]:
def plot_bars(x, title):
    fig = px.bar(x)
    
    fig.data[0].marker.color="white"
    fig.data[1].marker.color="green"
    fig.data[0].marker.line.width=0
    fig.data[1].marker.line.width=0
    
    fig.update_layout(
        barmode="group", 
        bargroupgap=0.0,
        template="plotly_dark",
        xaxis_title="K",
        yaxis_title="Value",
        title_text=title
    )
    
    fig.show()

In [ ]:
def get_crossval_report(model, X_train, Y_train):
    cv_scores = cross_validate(
        model, 
        X_train, Y_train, 
        cv=KFold(n_splits=5, random_state=SEED, shuffle=True),
        scoring=["r2", "neg_mean_absolute_error"],
        return_train_score=True
    )
    
    r2_scores = pd.DataFrame({
        "Train": cv_scores["train_r2"],
        "Test": cv_scores["test_r2"]
    })
    
    plot_bars(r2_scores*100, title="R^2")
    
    mae_scores = pd.DataFrame({
        "Train": cv_scores["train_neg_mean_absolute_error"],
        "Test": cv_scores["test_neg_mean_absolute_error"]
    })
    
    plot_bars(-1*mae_scores, title="MAE")

## Линейная модель

In [ ]:
linear_model = LinearRegression()

In [ ]:
get_crossval_report(linear_model, X_train, Y_train)

## Градиентный бустинг

In [ ]:
catboost_model = CatBoostRegressor(
    iterations=250,
    max_depth=2,
    random_seed=SEED,
    verbose=0
)

In [ ]:
get_crossval_report(catboost_model, X_train, Y_train)

## Финальная модель

In [ ]:
catboost_model.fit(X_train, Y_train)

In [ ]:
catboost_model.get_feature_importance(prettified=True).head(20)

## Метрики (отложенная выборка)

In [ ]:
test_prediction = catboost_model.predict(X_test)

In [ ]:
r2_score(Y_test, test_prediction) * 100

In [ ]:
mean_absolute_error(Y_test, test_prediction)

In [ ]:
root_mean_squared_error(Y_test, test_prediction)

In [ ]:
# Baseline
baseline_prediction = Y_train.mean() * np.ones(len(Y_test))

In [ ]:
mean_absolute_error(Y_test, baseline_prediction)

In [ ]:
root_mean_squared_error(Y_test, baseline_prediction)